In [15]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

In [16]:
!gdown --folder 1rDTQgHHL54qyhOz6IkLqEhKjhE6H5LIw

Retrieving folder contents
Processing file 186kClKhtmcvWMaJZp7NzFF7WgKlyGMTx cleaned_order_items.csv
Processing file 19-8hMEGK_FV0OY6E0fpqZ_WH6omKIWnu cleaned_orders.csv
Processing file 1yVay0sW__dnR8cNIJ-ApR_WrWJ8AYSEl cleaned_payments.csv
Processing file 1qstNYGS_k9hSMEs2vr-eqgxnGIKLiKAn cleaned_returns.csv
Processing file 1P-jOS_cDK6BMD25OmUv__y9pWIsAksbQ cleaned_reviews.csv
Processing file 18mG3bGPaf5RrmPbXYjRjgyn6yW_JRjCt cleaned_shipments.csv
Processing file 16P_nvcuuduQnA3j-AELWlk3VZgNMPeac customers_cleaned.csv
Processing file 12XPuEhvbMG8NKg5-80cWV82VM8rGT_HZ geography_cleaned.csv
Processing file 1W9tkL71XLedgt0VM119rInxy9ERY9eCs inventory_cleaned.csv
Processing file 136t_imY6EVZEh04x7whPlZwoo0tQIoDZ merge.csv
Processing file 16pvDJENj9ku1uRRqhtN_zuzpdsH20joq products_cleaned.csv
Processing file 1-yXL0SSJBPaUef7p6sbqG_v_YnONhMpG promotions_cleaned.csv
Processing file 1zVz6Qw4d8OptGkxZQpQuD5hyOqp63lXn sales_cleaned.csv
Processing file 1UAt4AnkSVD-b5AyLWhpzDTfdzuTg4P3w web_traff

In [17]:
import os
import pandas as pd

data_dir = '/content/cleaned_datasets/'
df = {}

# Kiểm tra thư mục tồn tại trước khi liệt kê file
if not os.path.exists(data_dir):
    print(f"Error: The directory '{data_dir}' was not found.")
    print("Please ensure the data download step (gdown) completed successfully.")
else:
    files = [f for f in os.listdir(data_dir) if f.endswith('.csv')]
    print(f'Loading {len(files)} files...')

    for file in files:
        file_path = os.path.join(data_dir, file)

        # Bỏ .csv + bỏ prefix 'cleaned_' hoặc suffix '_cleaned'
        df_name = os.path.splitext(file)[0]
        df_name = df_name.replace('cleaned_', '').replace('_cleaned', '')

        df[df_name] = pd.read_csv(file_path)
        print(f'- Loaded: {df_name} ({len(df[df_name])} rows)')

    print("\nAvailable datasets:", list(df.keys()))

    # Giờ gọi bình thường
    if 'customers' in df:
        display(df['customers'].head())

Loading 14 files...
- Loaded: sales (3833 rows)
- Loaded: inventory (60247 rows)
- Loaded: reviews (113551 rows)
- Loaded: promotions (50 rows)
- Loaded: web_traffic (3652 rows)


/tmp/ipykernel_6810/934092693.py:22: DtypeWarning: Columns (6) have mixed types. Specify dtype option on import or set low_memory=False.
  df[df_name] = pd.read_csv(file_path)


- Loaded: order_items (714669 rows)


/tmp/ipykernel_6810/934092693.py:22: DtypeWarning: Columns (0,12,13,14,17,20,22,23,24,25,31,32,33,35,36,37,38,41,42,44,50,52,53,54,55,56,57,59,60,62,63,64,65,68,69,70,73,74) have mixed types. Specify dtype option on import or set low_memory=False.
  df[df_name] = pd.read_csv(file_path)


- Loaded: merge (2960100 rows)
- Loaded: shipments (566067 rows)
- Loaded: geography (39948 rows)
- Loaded: payments (646945 rows)
- Loaded: products (2324 rows)
- Loaded: customers (121930 rows)
- Loaded: returns (39939 rows)
- Loaded: orders (646945 rows)

Available datasets: ['sales', 'inventory', 'reviews', 'promotions', 'web_traffic', 'order_items', 'merge', 'shipments', 'geography', 'payments', 'products', 'customers', 'returns', 'orders']


,customer_id,zip,city,signup_date,gender,age_group,acquisition_channel
0,1,15201,Hai Phong,2021-12-30,Female,35-44,social_media
1,2,15201,Hai Phong,2013-12-27,Female,45-54,email_campaign
2,3,15201,Hai Phong,2018-07-24,Female,18-24,organic_search
3,4,15201,Hai Phong,2017-11-29,Male,35-44,referral
4,5,15201,Hai Phong,2022-09-23,Male,55+,organic_search


In [18]:
import pandas as pd

# Gán tên biến cho từng file từ dictionary df
reviews = df['reviews']
sales = df['sales']
geography = df['geography']
web_traffic = df['web_traffic']
products = df['products']
customers = df['customers']
promotions = df['promotions']
inventory = df['inventory']
orders = df['orders']
payments = df['payments']
order_items = df['order_items']
returns = df['returns']
shipments = df['shipments']

# Chuyển đổi định dạng ngày tháng để tính toán
orders['order_date'] = pd.to_datetime(orders['order_date'])
shipments['ship_date'] = pd.to_datetime(shipments['ship_date'])
shipments['delivery_date'] = pd.to_datetime(shipments['delivery_date'])
returns['return_date'] = pd.to_datetime(returns['return_date'])
reviews['review_date'] = pd.to_datetime(reviews['review_date'])



In [19]:
# =========================================================
# 0. PRE-AGGREGATION (TRÁNH DUPLICATE KHI JOIN)
# =========================================================

# --- RETURNS: gom về 1 dòng / (order_id, product_id)
returns_agg = (
    returns
    .groupby(['order_id', 'product_id'], as_index=False)
    .agg(
        return_qty=('return_quantity', 'sum'),      # tổng số lượng trả
        refund_amount=('refund_amount', 'sum')      # tổng tiền refund
    )
)

# Tạo cờ có return hay không
returns_agg['is_returned'] = (returns_agg['return_qty'] > 0).astype(int)


# --- REVIEWS: gom về 1 dòng / (order_id, product_id)
reviews_agg = (
    reviews
    .groupby(['order_id', 'product_id'], as_index=False)
    .agg(
        rating=('rating', 'mean'),                 # rating trung bình
        review_count=('rating', 'count')           # số lượng review
    )
)


# =========================================================
# 1. BUILD MAIN FACT TABLE (ITEM-LEVEL)
# =========================================================

df = (
    order_items

    # -----------------------------------------------------
    # JOIN: ORDERS (thông tin ngữ cảnh đơn hàng)
    # -----------------------------------------------------
    .merge(
        orders[['order_id', 'customer_id', 'order_date',
                'device_type', 'order_source', 'zip', 'order_status', 'payment_method']], # Added 'payment_method'
        on='order_id',
        how='left'
    )

    # -----------------------------------------------------
    # JOIN: CUSTOMERS (demographics)
    # -----------------------------------------------------
    .merge(
        customers[['customer_id', 'gender', 'age_group', 'acquisition_channel']],
        on='customer_id',
        how='left'
    )

    # -----------------------------------------------------
    # JOIN: GEOGRAPHY (region)
    # -----------------------------------------------------
    .merge(
        geography[['zip', 'region']],
        on='zip',
        how='left'
    )

    # -----------------------------------------------------
    # JOIN: PRODUCTS (để tính cost + phân tích category)
    # -----------------------------------------------------
    .merge(
        products[['product_id', 'category', 'segment', 'cogs']],
        on='product_id',
        how='left'
    )

    # -----------------------------------------------------
    # JOIN: RETURNS (đã pre-agg)
    # -----------------------------------------------------
    .merge(
        returns_agg,
        on=['order_id', 'product_id'],
        how='left'
    )

    # -----------------------------------------------------
    # JOIN: REVIEWS
    # -----------------------------------------------------
    .merge(
        reviews_agg,
        on=['order_id', 'product_id'],
        how='left'
    )

    # -----------------------------------------------------
    # JOIN: SHIPMENTS (logistics)
    # -----------------------------------------------------
    .merge(
        shipments[['order_id', 'ship_date', 'delivery_date', 'shipping_fee']],
        on='order_id',
        how='left'
    )

    # -----------------------------------------------------
    # JOIN: PAYMENTS (order-level → chỉ reference)
    # -----------------------------------------------------
    .merge(
        payments[['order_id', 'payment_value']], # Removed 'payment_method' as it's now from orders
        on='order_id',
        how='left'
    )

    # -----------------------------------------------------
    # JOIN: PROMOTIONS (2 promo slots)
    # -----------------------------------------------------
    .merge(
        promotions.add_prefix('p1_'),
        left_on='promo_id',
        right_on='p1_promo_id',
        how='left'
    )
    .merge(
        promotions.add_prefix('p2_'),
        left_on='promo_id_2',
        right_on='p2_promo_id',
        how='left'
    )
)

In [20]:
# Check số rows sau khi join
print(f"Số dòng ban đầu: {len(order_items)}")
print(f"Số dòng sau khi merge: {len(df)}")

Số dòng ban đầu: 714669
Số dòng sau khi merge: 714669


In [21]:
df

,order_id,product_id,quantity,unit_price,discount_amount,promo_id,promo_id_2,customer_id,order_date,device_type,...,p2_promo_id,p2_promo_name,p2_promo_type,p2_discount_value,p2_start_date,p2_end_date,p2_applicable_category,p2_promo_channel,p2_stackable_flag,p2_min_order_value
0,1,2400,7,1138.22,0.0,NaN,NaN,58578,2012-07-04,desktop,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,609,7,10166.25,0.0,NaN,NaN,58621,2012-07-04,mobile,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,396,3,11220.33,0.0,NaN,NaN,58811,2012-07-04,desktop,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,635,5,10639.25,0.0,NaN,NaN,59453,2012-07-04,desktop,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,6,1935,1,1597.84,0.0,NaN,NaN,57821,2012-07-06,mobile,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
714664,834372,690,8,4473.92,0.0,NaN,NaN,19490,2022-12-31,mobile,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
714665,834377,1995,7,5250.79,0.0,NaN,NaN,73046,2022-12-31,mobile,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
714666,834387,2331,8,7389.06,0.0,NaN,NaN,107723,2022-12-31,mobile,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
714667,834392,1115,5,4767.33,0.0,NaN,NaN,139431,2022-12-31,desktop,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [26]:
df.shape

(714669, 52)

Lưu DataFrame `df` vào CSV

In [29]:
# Define the output path for Google Drive (if not already defined)
drive_output_path = '/content/drive/MyDrive/vinuni_datathon2026/cleaned_datasets/union.csv'

# Ensure the directory exists before saving
import os
drive_output_dir = os.path.dirname(drive_output_path)
if not os.path.exists(drive_output_dir):
    os.makedirs(drive_output_dir)
    print(f"Created directory: {drive_output_dir}")

# Save the DataFrame to CSV
df.to_csv(drive_output_path, index=False)

print(f"Đã lưu file vào Drive thành công: {drive_output_path}")

Đã lưu file vào Drive thành công: /content/drive/MyDrive/vinuni_datathon2026/cleaned_datasets/union.csv


In [33]:
!ls /content/drive/MyDrive/vinuni_datathon2026/cleaned_datasets

uninon.csv  union.csv


In [35]:
!ls /content/drive/MyDrive

vinuni_datathon2026
